In [1]:
from pathlib import Path
import sys
import ast
import pandas as pd
import numpy as np

PROJECT_DIR = Path.cwd().parent
SCRIPTS_DIR = PROJECT_DIR / "scripts"
DATA_PROCESSED_DIR = PROJECT_DIR / "data_processed"
MODELS_DIR = PROJECT_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))

from supervised_models import (
    SupervisedConfig,
    train_test_split_df,
    fit_transform_text,
    build_numeric_features,
    combine_features,
    build_model,
    evaluate_binary_classifier,
    save_model_and_vectorizer,
)


In [2]:
df = pd.read_csv(DATA_PROCESSED_DIR / "reddit_with_toxicity.csv")

# Reconstruir listas desde CSV
for col in ["tokens", "tokens_no_stop"]:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

df.head()


,body,created_utc,score,created_dt,date,year,month,year_month,text_clean,tokens,...,vad_arousal_mean,vad_dominance_mean,tox_word_count,tox_ratio,tox_has_toxic,tox_adv_token_count,tox_adv_score_raw,tox_adv_score_norm,tox_adv_max_span_score,tox_adv_has_toxic
0,Done - your wish is our command. Graves shall...,2017-04-01 16:01:43,2022,2017-04-01 16:01:43,2017-04-01,2017,4,2017-04,done your wish is our command graves shall onc...,"[done, your, wish, is, our, command, graves, s...",...,0.109800,0.097600,0,0.0,0,0,0.0,0.0,0.0,False
1,I don't play league but goddam it if I'm not g...,2017-04-01 03:56:12,4183,2017-04-01 03:56:12,2017-04-01,2017,4,2017-04,i don t play league but goddam it if i m not g...,"[i, don, t, play, league, but, goddam, it, if,...",...,0.068222,0.133444,0,0.0,0,0,0.0,0.0,0.0,False
2,http://imgur.com/a/8F7c6 WE DID IT REDDIT!,2017-04-01 04:01:06,267,2017-04-01 04:01:06,2017-04-01,2017,4,2017-04,http imgur com a 8f7c6 we did it reddit,"[http, imgur, com, a, 8f7c6, we, did, it, reddit]",...,0.667000,0.000000,0,0.0,0,0,0.0,0.0,0.0,False
3,Seems legit,2017-04-01 00:55:50,978,2017-04-01 00:55:50,2017-04-01,2017,4,2017-04,seems legit,"[seems, legit]",...,-0.146000,0.698000,0,0.0,0,0,0.0,0.0,0.0,False
4,"Graves no cigar, graves in peril, not to worry...",2017-04-01 01:19:17,670,2017-04-01 01:19:17,2017-04-01,2017,4,2017-04,graves no cigar graves in peril not to worry w...,"[graves, no, cigar, graves, in, peril, not, to...",...,0.432667,-0.246000,0,0.0,0,0,0.0,0.0,0.0,False


In [3]:
if "text_clean" in df.columns:
    df["text_for_model"] = df["text_clean"].astype(str)
else:
    df["text_for_model"] = df["tokens_no_stop"].apply(lambda toks: " ".join(toks))


In [4]:
def label_from_toxicity(row, low=0.01, high=0.05):
    """
    Zona gris entre low y high -> devuelve None.
    """
    score = row["tox_adv_score_norm"]
    if score >= high:
        return 1
    elif score <= low:
        return 0
    else:
        return None

df["label_toxic"] = df.apply(label_from_toxicity, axis=1)
df_labels = df.dropna(subset=["label_toxic"]).copy()
df_labels["label_toxic"] = df_labels["label_toxic"].astype(int)

df_labels["label_toxic"].value_counts(normalize=True)


label_toxic
0    0.917932
1    0.082068
Name: proportion, dtype: float64

In [6]:
numeric_cols = [
    "vad_valence_mean",
    "vad_arousal_mean",
    "vad_dominance_mean",
    "tox_adv_score_norm",
    "tox_word_count",
    "n_tokens_no_stop",
]

for col in numeric_cols:
    print(col, col in df_labels.columns)


vad_valence_mean True
vad_arousal_mean True
vad_dominance_mean True
tox_adv_score_norm True
tox_word_count True
n_tokens_no_stop True


In [7]:
cfg = SupervisedConfig(
    test_size=0.2,
    random_state=42,
    max_features=20000,
    min_df=5,
    max_df=0.9,
    ngram_range=(1, 2),
    model_type="logreg",  # luego probamos "svm" y "nb"
)

train_df, test_df = train_test_split_df(
    df_labels,
    label_col="label_toxic",
    test_size=cfg.test_size,
    random_state=cfg.random_state,
)

len(train_df), len(test_df)


(20909, 5228)

In [8]:
X_train_text, X_test_text, vectorizer = fit_transform_text(
    train_df["text_for_model"].tolist(),
    test_df["text_for_model"].tolist(),
    cfg,
)

X_train_num = build_numeric_features(train_df, numeric_cols)
X_test_num = build_numeric_features(test_df, numeric_cols)

X_train = combine_features(X_train_text, X_train_num)
X_test = combine_features(X_test_text, X_test_num)

y_train = train_df["label_toxic"].values
y_test = test_df["label_toxic"].values

X_train.shape, X_test.shape


((20909, 15212), (5228, 15212))

In [9]:
cfg.model_type = "logreg"  # probamos primero LogReg

model = build_model(cfg)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

metrics = evaluate_binary_classifier(y_test, y_pred)
metrics["accuracy"], metrics["f1"]
print(metrics["report"])


              precision    recall  f1-score   support

           0      1.000     0.998     0.999      4799
           1      0.979     0.998     0.988       429

    accuracy                          0.998      5228
   macro avg      0.990     0.998     0.994      5228
weighted avg      0.998     0.998     0.998      5228



In [11]:
for model_type in ["logreg", "svm", "nb"]:
    print("\n=== Modelo:", model_type, "===")
    cfg.model_type = model_type
    model = build_model(cfg)

    if model_type == "nb":
        # Naive Bayes SOLO con TF-IDF (sin features numéricas)
        Xtr, Xte = X_train_text, X_test_text
    else:
        # LogReg y SVM usan texto + features numéricas
        Xtr, Xte = X_train, X_test

    model.fit(Xtr, y_train)
    y_pred = model.predict(Xte)
    metrics = evaluate_binary_classifier(y_test, y_pred)
    print("Acc:", metrics["accuracy"], "F1:", metrics["f1"])
    print(metrics["report"])



=== Modelo: logreg ===
Acc: 0.9980872226472839 F1: 0.9884526558891455
              precision    recall  f1-score   support

           0      1.000     0.998     0.999      4799
           1      0.979     0.998     0.988       429

    accuracy                          0.998      5228
   macro avg      0.990     0.998     0.994      5228
weighted avg      0.998     0.998     0.998      5228


=== Modelo: svm ===
Acc: 0.999043611323642 F1: 0.9941927990708479
              precision    recall  f1-score   support

           0      1.000     0.999     0.999      4799
           1      0.991     0.998     0.994       429

    accuracy                          0.999      5228
   macro avg      0.995     0.998     0.997      5228
weighted avg      0.999     0.999     0.999      5228


=== Modelo: nb ===
Acc: 0.9349655700076511 F1: 0.3436293436293436
              precision    recall  f1-score   support

           0      0.934     1.000     0.966      4799
           1      1.000     0.20

In [13]:
cfg.model_type = "svm"   # o "logreg", el que mejor F1 te haya dado
best_model = build_model(cfg)
best_model.fit(X_train, y_train)  # este sí con texto + numéricas

from supervised_models import save_model_and_vectorizer

model_path = MODELS_DIR / "toxicity_classifier_svm.joblib"
save_model_and_vectorizer(
    best_model,
    vectorizer,
    numeric_cols,
    str(model_path),
)

print(" Modelo guardado en", model_path)


 Modelo guardado en c:\Users\lafp0\Documents\Github\ciencia-de-datos\7° semestre\procesamiento-lenguaje-natural\proyecto\models\toxicity_classifier_svm.joblib
